In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    os.getenv("NBA_DATABASE_URL")
)

OUT_TABLE = "fh_daily_ml_2023_onward_clean"
CSV_PATH  = r"C:\Users\padre\MLB_Code_2024\nba_1st_half\fh_daily_ml_2023_onward_clean.csv"

print("🔌 Connected to Postgres")

# ------------------------------------------------------------
# BUILD CLEAN DAILY ML TABLE WITH PRE-GAME EDGES
# ------------------------------------------------------------
sql = f"""
DROP TABLE IF EXISTS {OUT_TABLE};

CREATE TABLE {OUT_TABLE} AS
WITH base AS (
    SELECT
        game_id,
        game_date,
        team,
        opponent,
        site AS split,
        pts_scored_1,
        pts_allowed,
        margin,
        CASE
            WHEN EXTRACT(MONTH FROM game_date) >= 10
            THEN EXTRACT(YEAR FROM game_date)::int
            ELSE (EXTRACT(YEAR FROM game_date)::int - 1)
        END AS season,
        CASE
            WHEN margin > 0 THEN 1.0
            WHEN margin = 0 THEN 0.5
            ELSE 0.0
        END AS win_value
    FROM first_half_basics
),

features AS (
    SELECT
        b.*,

        -- rolling PRIOR 5 (pre-game only)
        AVG(pts_scored_1) OVER w AS r5_pts_prev,
        AVG(pts_allowed)  OVER w AS r5_allowed_prev,
        AVG(margin)       OVER w AS r5_margin_prev,
        AVG(win_value)    OVER w AS r5_wpct_prev,
        COUNT(*)          OVER w AS r5_n_prev

    FROM base b
    WINDOW w AS (
        PARTITION BY team, split, season
        ORDER BY game_date
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    )
),

season_2023_start AS (
    SELECT MIN(game_date) AS start_date
    FROM base
    WHERE season = 2023
),

games AS (
    SELECT
        h.game_date,
        h.game_id,

        h.team AS home_team,
        a.team AS away_team,

        -- HOME FEATURES
        ROUND(h.r5_pts_prev::numeric, 2)     AS home_r5_pts,
        ROUND(h.r5_allowed_prev::numeric, 2) AS home_r5_allowed,
        ROUND(h.r5_margin_prev::numeric, 2)  AS home_r5_margin,
        ROUND(h.r5_wpct_prev::numeric, 3)    AS home_wpct,
        h.r5_n_prev                          AS home_r5_n,

        -- AWAY FEATURES
        ROUND(a.r5_pts_prev::numeric, 2)     AS away_r5_pts,
        ROUND(a.r5_allowed_prev::numeric, 2) AS away_r5_allowed,
        ROUND(a.r5_margin_prev::numeric, 2)  AS away_r5_margin,
        ROUND(a.r5_wpct_prev::numeric, 3)    AS away_wpct,
        a.r5_n_prev                          AS away_r5_n,

        -- PRE-GAME EDGES
        ROUND((h.r5_margin_prev - a.r5_margin_prev)::numeric, 2)
            AS home_plus_minus,

        ROUND(
            CASE
                WHEN (ABS(h.r5_margin_prev) + ABS(a.r5_margin_prev)) > 0
                THEN (h.r5_margin_prev - a.r5_margin_prev)
                     / (ABS(h.r5_margin_prev) + ABS(a.r5_margin_prev))
                ELSE NULL
            END
        , 3) AS pct_margin,

        -- TARGETS
        h.pts_scored_1 AS home_score,
        a.pts_scored_1 AS away_score,
        (h.pts_scored_1 - a.pts_scored_1) AS margin,

        CASE
            WHEN (h.pts_scored_1 - a.pts_scored_1) > 0 THEN 1.0
            WHEN (h.pts_scored_1 - a.pts_scored_1) = 0 THEN 0.5
            ELSE 0.0
        END AS home_win

    FROM features h
    JOIN features a
      ON h.game_id = a.game_id
     AND h.split = 'H'
     AND a.split = 'A'
),

filtered AS (
    SELECT *
    FROM games g
    CROSS JOIN season_2023_start s
    WHERE g.game_date >= s.start_date
      AND g.home_r5_n >= 1
      AND g.away_r5_n >= 1
)

SELECT *
FROM filtered
ORDER BY game_date, game_id;
"""

with engine.begin() as conn:
    print(f"🧱 Building ML table with edges: {OUT_TABLE}")
    conn.execute(text(sql))

print(f"✅ Table created: {OUT_TABLE}")

# ------------------------------------------------------------
# SANITY CHECK
# ------------------------------------------------------------
with engine.connect() as conn:
    n = conn.execute(text(f"SELECT COUNT(*) FROM {OUT_TABLE};")).scalar()
print(f"📦 Rows in {OUT_TABLE}: {n}")

sample = pd.read_sql(
    text(f"SELECT * FROM {OUT_TABLE} ORDER BY game_date DESC, game_id DESC LIMIT 10;"),
    engine
)
print(sample.to_string(index=False))

# ------------------------------------------------------------
# EXPORT CSV
# ------------------------------------------------------------
print(f"📁 Exporting CSV to: {CSV_PATH}")
full_df = pd.read_sql(
    text(f"SELECT * FROM {OUT_TABLE} ORDER BY game_date, game_id;"),
    engine
)
full_df.to_csv(CSV_PATH, index=False)
print("✅ CSV export complete")


🔌 Connected to Postgres
🧱 Building ML table with edges: fh_daily_ml_2023_onward_clean
✅ Table created: fh_daily_ml_2023_onward_clean
📦 Rows in fh_daily_ml_2023_onward_clean: 3264
 game_date  game_id home_team away_team  home_r5_pts  home_r5_allowed  home_r5_margin  home_wpct  home_r5_n  away_r5_pts  away_r5_allowed  away_r5_margin  away_wpct  away_r5_n  home_plus_minus  pct_margin  home_score  away_score  margin  home_win start_date
2026-02-01 18447529       DEN       OKC         54.8             53.6             1.2        0.6          5         61.0             54.6             6.4        0.8          5             -5.2      -0.684          55          62      -7       0.0 2023-10-24
2026-02-01 18447528       POR       CLE         58.4             59.0            -0.6        0.4          5         54.6             48.2             6.4        0.6          5             -7.0      -1.000          48          57      -9       0.0 2023-10-24
2026-02-01 18447527       PHX       LAC        